# Pipeline Diagnostics Usage

This notebook demonstrates the new offline CSI pipeline diagnostic helpers. Unlike the general getting-started tutorials, this notebook does not train models or run the main pipeline; it only screens candidate CSI preprocessing offline before heavier validation.

Each code cell is a self-contained module:

1. Environment setup and imports
2. Synthetic CSI sample generation
3. Candidate preprocessing pipeline
4. Full diagnostics artifact generation
5. Metrics-only comparison
6. Plot-only comparison
7. Batch diagnostics and appended reports
8. Lightweight decision rules
9. Replacing synthetic arrays with real CSI

The examples run on CPU, do not start services, and use synthetic CSI by default.


In [ ]:
# Module 1: environment setup and imports
from pathlib import Path
import sys

# Make the notebook work both from the repo root and from examples/.
for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import csv
import json
import numpy as np
import matplotlib.pyplot as plt

from wsdp.diagnostics import (
    compute_pipeline_metrics,
    plot_pipeline_diagnostics,
    run_pipeline_diagnostics,
)

output_dir = Path("outputs/pipeline_diagnostics_demo")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Diagnostics will be written to: {output_dir.resolve()}")


In [ ]:
# Module 2: create a synthetic complex CSI sample
# Expected CSI shape is (time, subcarriers) or (time, subcarriers, antennas).
rng = np.random.default_rng(7)
T, F, A = 256, 30, 3
sampling_rate = 100.0  # Hz
motion_freq_hz = 3.0

time = np.arange(T) / sampling_rate
subcarrier_weight = np.linspace(0.6, 1.4, F)[None, :, None]
antenna_weight = np.linspace(1.0, 1.3, A)[None, None, :]

static_path = np.ones((T, F, A), dtype=complex)
motion_path = 0.35 * np.exp(1j * 2 * np.pi * motion_freq_hz * time)[:, None, None]
noise = 0.03 * (rng.standard_normal((T, F, A)) + 1j * rng.standard_normal((T, F, A)))

raw_csi = static_path + motion_path * subcarrier_weight * antenna_weight + noise
print("raw_csi shape:", raw_csi.shape)


In [ ]:
# Module 3: define a candidate preprocessing pipeline
# Replace this function with your own CSI preprocessing pipeline.
def moving_average_time(csi, window=5):
    kernel = np.ones(window) / window
    return np.apply_along_axis(lambda x: np.convolve(x, kernel, mode="same"), axis=0, arr=csi)

processed_csi = moving_average_time(raw_csi, window=5)

# A deliberately bad baseline helps calibrate what over-smoothing looks like.
over_smoothed_csi = np.repeat(raw_csi.mean(axis=0, keepdims=True), raw_csi.shape[0], axis=0)

print("processed_csi shape:", processed_csi.shape)
print("over_smoothed_csi shape:", over_smoothed_csi.shape)


In [ ]:
# Module 4: run full diagnostics and generate artifacts
result = run_pipeline_diagnostics(
    raw_csi,
    processed_csi,
    output_dir=output_dir,
    sample_name="gesture_demo",
    include_doppler=True,
    sampling_rate=sampling_rate,
    motion_band=(0.5, 10.0),
    antenna_idx=0,
    n_fft=64,
    hop_length=32,
    pipeline_name="moving_average_time",
    pipeline_config={"window": 5},
)

print("diagnostic_path:", result["diagnostic_path"])
print("metrics_path:", result["metrics_path"])
print("manifest_path:", result["manifest_path"])
print(json.dumps(result["metrics"], indent=2))


In [ ]:
# Module 5: compute metrics only, without writing PNG/CSV/manifest artifacts
candidates = {
    "moving_average_time": processed_csi,
    "over_smoothed": over_smoothed_csi,
}

for name, candidate in candidates.items():
    metrics = compute_pipeline_metrics(
        raw_csi,
        candidate,
        sampling_rate=sampling_rate,
        motion_band=(0.5, 10.0),
    )
    print(f"\n{name}")
    print("  signal_preservation_ratio:", round(metrics["signal_preservation_ratio"], 4))
    print("  motion_band_energy_ratio_delta:", round(metrics["motion_band_energy_ratio_delta"], 4))
    print("  mean_abs_difference:", round(metrics["mean_abs_difference"], 4))


In [ ]:
# Module 6: plot only, useful inside notebooks when you do not need CSV/manifest files
plot_path = output_dir / "gesture_demo_plot_only.png"
fig = plot_pipeline_diagnostics(
    raw_csi,
    processed_csi,
    save_path=plot_path,
    include_doppler=False,
    antenna_idx=0,
)
print("plot_path:", plot_path)
fig


In [ ]:
# Module 7: batch diagnostics; metrics.csv and manifest.json append one row/sample per call
for idx, window in enumerate((3, 7, 11), start=1):
    candidate = moving_average_time(raw_csi, window=window)
    run_pipeline_diagnostics(
        raw_csi,
        candidate,
        output_dir=output_dir,
        sample_name=f"gesture_demo_window_{window}",
        include_doppler=False,
        sampling_rate=sampling_rate,
        motion_band=(0.5, 10.0),
        antenna_idx=0,
        pipeline_name="moving_average_time",
        pipeline_config={"window": window},
    )

with (output_dir / "metrics.csv").open(newline="") as f:
    rows = list(csv.DictReader(f))
print("metrics rows:", len(rows))
print("latest samples:", [row["sample"] for row in rows[-3:]])


In [ ]:
# Module 8: KISS decision rules for early pipeline rejection
# These are not final accuracy guarantees; they are cheap early filters.
def quick_screen(metrics):
    signal_ratio = metrics["signal_preservation_ratio"]
    motion_delta = metrics["motion_band_energy_ratio_delta"]
    if signal_ratio < 0.2:
        return "reject: likely over-smoothed motion dynamics"
    if signal_ratio > 5.0:
        return "inspect: likely noise amplification or scale mismatch"
    if motion_delta < -0.2:
        return "inspect: target motion band was weakened"
    return "pass early screen: continue to task-level validation"

for name, candidate in candidates.items():
    metrics = compute_pipeline_metrics(
        raw_csi,
        candidate,
        sampling_rate=sampling_rate,
        motion_band=(0.5, 10.0),
    )
    print(f"{name}: {quick_screen(metrics)}")


In [ ]:
# Module 9: replace synthetic arrays with your real CSI arrays
# Uncomment and edit the paths once you have saved raw/processed CSI as .npy files.
#
# raw_csi = np.load("/path/to/raw_csi.npy")
# processed_csi = np.load("/path/to/processed_csi.npy")
#
# result = run_pipeline_diagnostics(
#     raw_csi,
#     processed_csi,
#     output_dir="outputs/my_pipeline_diagnostics",
#     sample_name="real_sample_001",
#     include_doppler=True,
#     sampling_rate=100.0,
#     motion_band=(0.5, 10.0),
#     antenna_idx=0,
#     pipeline_name="my_pipeline",
#     pipeline_config={"replace": "with your config"},
# )
# result
